# 05 — Model B pilot: gloss → motion vs B0/B1, plus ablation T (Day 4)

Runs exactly the same code as `uv run python -m bdsl.cli train-gloss2motion` (see `CLAUDE.md` → COMMANDS).

- **Inputs:** G0 pseudo-glosses (`gloss_auto`, markers removed, plus G0 tag tokens; ADR-0014), or the normalised Bangla words with `--input text` (ablation T). Reviewed glosses are not used.
- **Targets:** pilot MediaPipe features (feat_ver `fa89892d4908`, ADR-0023): 9 upper-body points, both hands, 52 blendshapes, head rotation. Missing hands are masked, never zero-filled into the loss (gaps ≤ 5 frames interpolated). MediaPipe hand labels are used as given (G8); no clip is excluded (G9).
- **Baselines:** B0 = mean motion; B1 = the real motion of the most similar training sentence.
- **Pilot numbers are a "pipeline test"** (5 val / 5 test clips), never generalisation (ADR-0020).
- Outputs go to `work/runs/` (gitignored); the first seed also writes `predictions/<Index>.npz` for the Day 5 avatar.


In [ ]:
from bdsl.cli import main
from bdsl.config import load_config

cfg = load_config(data="pilot50")
print("env:", cfg["_meta"]["env"], "| runs:", cfg["paths"]["runs_dir"])
print(cfg["gloss2motion"])

In [ ]:
# Overfit checks first (1 clip, then 40).
print("exit code:", main(["train-gloss2motion", "--overfit", "1"]))
print("exit code:", main(["train-gloss2motion", "--overfit", "40"]))

In [ ]:
# Model B (gloss input) and ablation T (Bangla words), 3 seeds each, early stopping on val.
print("exit code:", main(["train-gloss2motion", "--input", "gloss"]))
print("exit code:", main(["train-gloss2motion", "--input", "text"]))

In [ ]:
# The newest gloss run: Model B vs B0/B1 per split, and the per-clip table.
import json
from pathlib import Path

import pandas as pd

run = max(Path(cfg["paths"]["runs_dir"]).glob("gloss2motion_gloss_pilot50_*"), key=lambda p: p.stat().st_mtime)
m = json.loads((run / "metrics.json").read_text(encoding="utf-8"))
for s in ("val", "test"):
    print(s, "B", {k: round(v["mean"], 3) for k, v in m["model_b_mean_sd"][s].items()})
    print(s, "B0", {k: round(v, 3) for k, v in m["B0"][s].items()})
    print(s, "B1", {k: round(v, 3) for k, v in m["B1"][s].items()})
pd.read_csv(run / "per_item.csv", encoding="utf-8-sig")